# GRAPE on a single spin

This in an introductory example to the [GRAPE](10.1016/j.jmr.2004.11.004) method and its implementation in this software package.

## 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator

First, let's generate a piecewise constant (PWC) pulse envelope for the Gaussian pulse. For GRAPE, we need to specify an anstanz for piecewise constant controls at a given resolution. Here, we setup a Gaussian initial guess, sampling at 21 points during a gate time of 20ns.

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 21)
tone = GaussEnvelope(amplitude=Quantity(np.pi / t_final / 3, -np.pi / t_final, np.pi / t_final))
tone.t_final.set_value(t_final)

gen = PWCGenerator(envelopes=[tone], tlist=tlist)
gen.multiply_flat_top = True
gen.max_amplitude = 2 * 1e8

We have added the option `multiplyFlatTop`, to ensure the pulse to start and end smoothly at 0 and `t_final`. This acts like the `FlatTopGaussianFilter`, but enforced directly by the `PWCGenerator`.

In [ ]:
from plotting import plot_signal

ts = np.linspace(0, t_final, 501)
fig, ax = plt.subplots(1, figsize=(5, 3))
plot_signal(tone, ts, ax, linestyle="-", label="Smooth")
plot_signal(gen, ts, ax, linestyle="--", label="PWC")
ax.legend(loc=1, frameon=True)
plt.show()

## 2. Define Hamiltonian in the rotating frame of drive

As a simple toy model, we use a single spin.

In [ ]:
from paraqeet import Array
from paraqeet.model.rotating_frame import RotatingFrameDrive
from paraqeet.model.schroedinger_equation import SchroedingerEquation
from paraqeet.model.system import OpenSystem


class SpinRWA(OpenSystem):
    """A Single Spin."""

    def __init__(self, drives=None):
        super().__init__(drives)
        self.sigma_p = np.array([[0j, 1], [0, 0]])
        self.dim = 2

    def get_hamiltonian_at_timestep(self, timestep: float) -> Array:
        """Just sigma-X."""
        return self._drives[0].get_value_at_timestep(self.sigma_p, timestep)

    def get_hamiltonian_and_gradient(self, times: Array) -> tuple[Array, Array] | tuple[float, Array]:
        """Gradient is just the drive matrix."""
        return self.get_hamiltonian(times), self._drives[0].get_gradient(self.sigma_p, times)

    def get_hamiltonian_gradient_at_timestep(self, time):
        """Computes the gradient"""
        return self._drives[0].get_gradient_at_timestep(self.sigma_p, time)

    def dimension(self) -> int:
        """Returns the dimension"""
        raise self.d

    def get_jump_operators(self) -> list[tuple[Array, Array]]:
        """Returns an empyt list since we study closed system dynamics"""
        return []

    def get_parameters(self):
        """Returns the parameters, which in this case are only the drive parameters"""
        return self._get_drive_parameters()


drive = RotatingFrameDrive(gen)
spin = SpinRWA(drives=[drive])
model = SchroedingerEquation(
    hamiltonian_func=spin.get_hamiltonian, hamiltonian_and_gradient_func=spin.get_hamiltonian_and_gradient
)

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.measurement.utils import overlap_state_vector
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE
from paraqeet.propagation.utils import grape_operator_sandwich_function_closed

init = np.array([[1.0], [0.0]])  # |0>
target = np.array([[0.0], [1.0]])  # |1>

prop = ScipyExpmGRAPE(
    eom_func=model.get_value,
    eom_and_grad_func=model.get_value_and_gradient,
    resolution=2e9,
    initial_state=init,
    target_state=target,
    operator_sandwich_function=grape_operator_sandwich_function_closed,
)

times = np.array([0.0, t_final])

zeroone = StateTransferFidelityGRAPE(
    propagation_func=prop.propagate,
    propagation_and_gradient_func=prop.get_value_and_gradient,
    target_state=target,
    overlap=overlap_state_vector,
)

In [ ]:
from plotting import plot_signal_and_dynamics

ts = np.linspace(0.0, t_final, 101)
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"]);

In [ ]:
zeroone.measure(times)

In [ ]:
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

optmap = OptimizationMap()
optmap.add(gen)
optmap.register_params_with_optimizables()

opt_grad = ScipyOptimizerGradient(measure_and_gradient_func=zeroone.get_value_and_gradient, optimization_map=optmap)

Unlike the previous examples, for GRAPE based optimization, we need to specify the exact time grid used to discretize the signal for optimization. This is required to compute the correct gradients at the exact time points. The time grid used for discretization can be found from a `PWCGenerator` using `gen.tlist`.

In [ ]:
opt_grad.optimize(gen.tlist)

For this simple example, we reach near perfect fidelity within a few iterations.

In [ ]:
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"]);

## With open system

Lets first reset the pulse and create a open-system model

In [ ]:
import numpy as np

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator

In [ ]:
tone = GaussEnvelope(amplitude=Quantity(np.pi / t_final / 3, -np.pi / t_final, np.pi / t_final))
tone.t_final.set_value(t_final)
gen = PWCGenerator(envelopes=[tone], tlist=tlist)
gen.multiply_flat_top = True
gen.max_amplitude = 5 * 1e8

In [ ]:
import jax.numpy as jnp

from paraqeet.model.master_equation import MasterEquation
from paraqeet.model.rotating_frame import RotatingFrameDrive
from paraqeet.model.system import OpenSystem


class SpinRWA(OpenSystem):
    """A Single Spin."""

    def __init__(self, drives=None):
        super().__init__(drives)
        self.sigma_p = np.array([[0j, 1], [0, 0]])
        self.sigma_m = np.array([[0j, 0], [1, 0]])
        self.sigma_z = np.array([[1, 0j], [0, -1]])
        self.dim = 2
        self.t1 = Quantity(10e-6, 1e-6, 100e-6)
        self.temp = Quantity(10e-3, 1e-3, 50e-3)
        self.t2star = Quantity(20e-6, 1e-6, 100e-6)

    def get_hamiltonian_at_timestep(self, timestep: float) -> Array:
        """Just sigma-X."""
        return self._drives[0].get_value_at_timestep(self.sigma_p, timestep)

    def get_hamiltonian_and_gradient(self, times: Array) -> tuple[Array, Array] | tuple[float, Array]:
        """Gradient is just the drive matrix."""
        return self.get_hamiltonian(times), self._drives[0].get_gradient(self.sigma_p, times)

    def get_hamiltonian_gradient_at_timestep(self, time):
        """Computes the gradient"""
        return self._drives[0].get_gradient_at_timestep(self.sigma_p, time)

    def dimension(self) -> int:
        """Returns the dimension"""
        raise self.d

    def get_parameters(self):
        """Returns the parameters, which in this case are only the drive parameters"""
        return self._get_drive_parameters()

    def get_decay_rates(self) -> list[float]:
        """Return decay rate for T1, T2star and Temp respectively."""
        if (self.t1 is None) or (self.t2star is None) or (self.temp is None):
            raise Exception("Specify values of T1, T2star and Temp for Open system simulations.")

        gamma = 1 / self.t1.get_value()
        gamma_t2star = 0.5 / self.t2star.get_value()

        hbar_over_kb = 7.638232582257738e-12
        beta = hbar_over_kb / (self.temp.get_value())
        # inserting typical qubit freq here. TODO - CHECK
        nbar = jnp.exp(-beta * 5e9)
        gamma_temp = gamma * nbar
        gamma_t1 = gamma * (nbar + 1)
        return [gamma_t1, gamma_temp, gamma_t2star]

    def get_jump_operators(self) -> list[jnp.ndarray]:
        """Return a list tuples of decay rates and jump operators for each subsystem."""
        gamma_t1, gamma_temp, gamma_t2star = self.get_decay_rates()
        col_t1 = jnp.sqrt(gamma_t1) * self.sigma_p
        col_temp = jnp.sqrt(gamma_temp) * self.sigma_m
        col_t2star = jnp.sqrt(gamma_t2star) * 2 * self.sigma_z
        return [col_t1, col_temp, col_t2star]


drive = RotatingFrameDrive(gen)
spin = SpinRWA(drives=[drive])
model = MasterEquation(
    hamiltonian_func=spin.get_hamiltonian,
    hamiltonian_and_gradient_func=spin.get_hamiltonian_and_gradient,
    jump_operators=spin.get_jump_operators(),
)

Lets test GRAPE with ODE-propgation 

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.measurement.utils import overlap_density_matrix
from paraqeet.propagation.utils import grape_operator_sandwich_function_open, lindblad_step, reverse_lindblad_step
from paraqeet.propagation.vern7_grape import Vern7GRAPE

init = np.array([[1.0], [0.0j]])  # |0>
target = np.array([[0.0j], [1.0]])  # |1>

init = jnp.matmul(init, init.T.conj())
target = jnp.matmul(target, target.T.conj())

prop = Vern7GRAPE(
    eom_func=model.get_eom_ode_propagation,
    eom_and_gradient_func=model.get_eom_and_gradient_ode_propagation,
    resolution=10e9,
    initial_state=init,
    target_state=target,
    step_function=lindblad_step,
    reverse_step_function=reverse_lindblad_step,
    operator_sandwich_function=grape_operator_sandwich_function_open,
    jump_operators=spin.get_jump_operators(),
)

zeroone = StateTransferFidelityGRAPE(
    propagation_func=prop.propagate,
    propagation_and_gradient_func=prop.get_value_and_gradient,
    target_state=target,
    overlap=overlap_density_matrix,
)

In [ ]:
init

In [ ]:
ts = np.linspace(0, t_final, 101)
plot_signal_and_dynamics(gen, prop, times=ts, state_labels=[r"$\rho_0$", r"$\rho_1$"], open_system=True);

In [ ]:
zeroone.measure(times)

In [ ]:
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

optmap = OptimizationMap()
optmap.add(gen)
optmap.register_params_with_optimizables()

In [ ]:
opt_grad = ScipyOptimizerGradient(measure_and_gradient_func=zeroone.get_value_and_gradient, optimization_map=optmap)
opt_grad.set_options({"disp": True})

In [ ]:
opt_grad.optimize(tlist)

In [ ]:
plot_signal_and_dynamics(gen, prop, times=ts, state_labels=[r"$\rho_0$", r"$\rho_1$"], open_system=True);

In [ ]:
zeroone.measure(times)